# Self-Supervised Learning for Images
### Rotation, SimCLR, and the Collapse Problem - Measured by Linear Probing

**The 'BERT for images' idea.** In NLP we pre-train a giant model on raw text with no human labels, then transfer it to downstream tasks with very little labeled data. Self-supervised learning (SSL) brings the same recipe to vision: learn a general-purpose **image encoder** from large amounts of *unlabeled* images by solving a **pretext task** we can score for free, then transfer that encoder to real tasks (classification, detection, segmentation) where labels are scarce.

**The one experiment this notebook builds toward.** We train a single small CNN encoder **three ways**:

1. **Random init** - no pre-training (the floor).
2. **Rotation prediction** - a *predictive* pretext task.
3. **SimCLR** - a *contrastive* pretext task.

Then we **freeze** each encoder and compare them with one shared yardstick - **linear-probe accuracy** on a small labeled set. Along the way we *watch feature collapse happen live* and see how bootstrapping methods (SimSiam / BYOL) avoid it.

**Demo philosophy.** One small CNN encoder, one small image subset (a few thousand CIFAR-10 images with labels withheld), free-GPU / CPU-friendly, end-to-end in a single class session. Everything is seeded and runs top-to-bottom with **Run All**.

## Learning objectives and roadmap

By the end of this notebook you will be able to:

- **LO1** - Explain the SSL framework (pretext -> transfer) and place the five SSL families on one map. *(C03-C04)*
- **LO2** - Use the **linear evaluation protocol** (freeze the encoder, train only a linear head) as the single yardstick, with random init as the baseline. *(C09-C10, reused throughout)*
- **LO3** - Implement and run **rotation prediction** and measure its probe gain over random init. *(C11-C14)*
- **LO4** - Implement **SimCLR** (positive pairs via augmentation, NT-Xent loss) and explore how augmentation defines the positive pair. *(C15-C18)*
- **LO5** - Explain **bootstrapping** (BYOL / SimSiam) and **feature collapse**, and demonstrate live that a naive Siamese net collapses while predictor + stop-gradient prevents it. *(C19-C21)*
- **LO6** - Synthesize a fair head-to-head comparison and probe the **label-scarcity** regime where SSL helps most. *(C22-C24)*

**Roadmap:** framework + taxonomy (C03-C04) -> setup and data (C05-C07) -> shared encoder + linear-eval yardstick (C08-C10) -> rotation pretext (C11-C14) -> contrastive SimCLR (C15-C18) -> bootstrapping and collapse (C19-C21) -> head-to-head synthesis + label-scarcity widget (C22-C24).

> The linear-eval protocol is introduced **early** (C09-C10) because we reuse it to score *every* encoder.

## 1 - The self-supervised learning framework

A **pretext task** invents a supervision signal *from the data itself* - no human labels required. We design a task whose answer is automatically known (e.g. 'by how much was this image rotated?'), and train an **encoder** to solve it. To do well, the encoder is forced to learn something about image structure, producing reusable **representations**.

```
encoder --> [pretext head]    --> pretext target   (training time, label-free)
encoder --> [downstream head] --> real labels       (transfer time, few labels)
```

The pattern is always **pretext-then-transfer**:

1. **Pre-train:** solve the pretext task on lots of unlabeled images -> a trained encoder.
2. **Transfer:** discard the pretext head, keep the encoder, attach a lightweight head for the real task.

How do we know the representation is any good? We **attach a simple head to the frozen encoder** and measure how well it does - the linear-evaluation protocol we formalize in C09. Every method in this notebook is just a different choice of pretext task plugged into this same frame.

## 2 - A taxonomy of image SSL: five families

| Family | Representative image methods | What it predicts / its signal | How it avoids trivial (collapsed) solutions | Run here? |
|---|---|---|---|---|
| **Generative** | image-GPT, masked autoencoders (MAE) | Reconstruct pixels / missing patches | Reconstruction is hard; a constant output has high error | extension |
| **Predictive** | **Rotation (RotNet)**, context / jigsaw, DeepCluster | A discrete property of the transform (which rotation? which arrangement?) | Cross-entropy on a non-trivial discrete label | **yes - rotation** |
| **Contrastive** | **SimCLR**, MoCo | Which two views came from the same image | **Negatives**: push other images apart | **yes - SimCLR** |
| **Bootstrapping** | **BYOL**, **SimSiam** | Match two views - *no negatives* | **Architectural asymmetry**: predictor + stop-gradient (+ EMA target) | **yes - collapse demo** |
| **Regularization** | Barlow Twins, VICReg | Match views | Explicit **variance / covariance** terms keep dimensions varied and decorrelated | extension |

We run **three** families hands-on - Predictive (rotation), Contrastive (SimCLR), and a Bootstrapping collapse demo - and leave the others as extensions (C24). Keep this table in mind: by C21 we will have *watched* the collapse-avoidance mechanism of one family and can map it back to all the others.

In [ ]:
# === C05 . Environment setup (the single dependency cell) ===
import importlib, subprocess, sys

WIDGETS_OK = True
try:
    import ipywidgets  # noqa: F401
except ImportError:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'], check=True)
        importlib.invalidate_caches()
        import ipywidgets  # noqa: F401
    except Exception as e:
        WIDGETS_OK = False
        print(f'[warn] ipywidgets unavailable ({e}); C16/C23 will render static defaults.')

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
try:
    import ipywidgets as widgets
except Exception:
    widgets = None
    WIDGETS_OK = False
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed: int = 42) -> None:
    '''Seed python / numpy / torch (incl. CUDA) for a reproducible Run-All.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

# --- sanity checks ---
assert isinstance(torch.__version__, str) and DEVICE.type in {'cuda', 'cpu'}
set_seed(7); _a = torch.randn(2)
set_seed(7); _b = torch.randn(2)
assert torch.allclose(_a, _b), 'set_seed is not reproducible'
set_seed(42)

print(f'torch {torch.__version__} | device: {DEVICE} | widgets: {WIDGETS_OK}')
if DEVICE.type == 'cpu':
    print('[note] No GPU detected. C17 (SimCLR) is the heaviest cell - reduce '
          'SIMCLR_EPOCHS / SIMCLR_BATCH or use a Colab free T4 GPU for speed.')

## Data: making SSL honest with an unlabeled / labeled split

Real SSL pre-trains on *huge unlabeled* image collections, then fine-tunes or probes with a *few* labels. To reproduce that contrast cheaply we use **CIFAR-10** but **deliberately hide its labels**:

- **`X_unlabeled`** - a few thousand images with **labels withheld**. This is our *pretext pool*: the only data the rotation / SimCLR / collapse training ever sees. **No labels are used here.**
- **`probe_train` / `probe_test`** - a small, **disjoint**, class-balanced labeled set used *only* by the linear probe to **measure** representation quality afterwards.

> Labels are **never** seen during pretext training - they exist only to grade the encoder once it is frozen. The C07 split draws the pretext pool and the probe set from **disjoint index sets**, so probe labels can never leak into pretext training. This mirrors the real regime where labels are scarce and expensive.

In [ ]:
# === C07 . Load CIFAR-10 once and build the unlabeled / labeled split ===
set_seed(42)

# Normalization convention (fixed here, reused everywhere):
#   * base images stored as float tensors in [0,1]
#   * transforms.Normalize(MEAN, STD) is applied ONCE, at encoder input
#   * augmentation happens in [0,1] BEFORE normalization
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
normalize = transforms.Normalize(MEAN, STD)

N_UNLABELED = 4000
N_PROBE_TRAIN_PER_CLASS = 50
N_PROBE_TEST_PER_CLASS = 100

try:
    train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
    test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)
except Exception as e:
    raise RuntimeError('CIFAR-10 download/load failed (' + str(e) + '). Check your network and '
                       'retry; if data is already in ./data you can set download=False.')

CLASS_NAMES = train_set.classes
train_imgs = torch.tensor(train_set.data).permute(0, 3, 1, 2).float().div(255)
train_lbls = torch.tensor(train_set.targets)
test_imgs = torch.tensor(test_set.data).permute(0, 3, 1, 2).float().div(255)
test_lbls = torch.tensor(test_set.targets)

def _to_display(img):
    '''Clamp a [0,1] CHW image and return HWC for plotting.'''
    return img.clamp(0, 1).permute(1, 2, 0).cpu()

def _sample_balanced(images, labels, per_class, allowed_idx, generator):
    '''Draw per_class class-balanced indices from allowed_idx (seeded).'''
    chosen = []
    for c in range(len(CLASS_NAMES)):
        cls_idx = allowed_idx[labels[allowed_idx] == c]
        take = min(per_class, cls_idx.numel())
        if take < per_class:
            print('[warn] class ' + str(c) + ': only ' + str(take) + ' available; clamped.')
        perm = torch.randperm(cls_idx.numel(), generator=generator)[:take]
        chosen.append(cls_idx[perm])
    chosen = torch.cat(chosen)
    return images[chosen], labels[chosen]

g = torch.Generator().manual_seed(42)
perm = torch.randperm(train_imgs.shape[0], generator=g)
unlabeled_idx = perm[:N_UNLABELED]
remaining_idx = perm[N_UNLABELED:]                      # disjoint from unlabeled_idx

X_unlabeled = train_imgs[unlabeled_idx].clone()         # labels intentionally discarded
probe_train = _sample_balanced(train_imgs, train_lbls, N_PROBE_TRAIN_PER_CLASS, remaining_idx, g)
g_test = torch.Generator().manual_seed(123)
probe_test = _sample_balanced(test_imgs, test_lbls, N_PROBE_TEST_PER_CLASS,
                              torch.arange(test_imgs.shape[0]), g_test)

# --- sanity checks ---
assert X_unlabeled.shape == (N_UNLABELED, 3, 32, 32) and X_unlabeled.dtype == torch.float32
assert X_unlabeled.min() >= 0 and X_unlabeled.max() <= 1
assert probe_train[0].shape[0] == 10 * N_PROBE_TRAIN_PER_CLASS
assert set(probe_train[1].tolist()) == set(range(10))
assert len(CLASS_NAMES) == 10
assert len(set(unlabeled_idx.tolist()) & set(remaining_idx.tolist())) == 0   # no leakage

print('X_unlabeled:', tuple(X_unlabeled.shape), '(labels withheld) | probe_train:',
      probe_train[0].shape[0], '| probe_test:', probe_test[0].shape[0])

# --- one labeled thumbnail per class ---
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for c, ax in enumerate(axes):
    idx = (probe_train[1] == c).nonzero(as_tuple=True)[0][0]
    ax.imshow(_to_display(probe_train[0][idx]))
    ax.set_title(CLASS_NAMES[c], fontsize=8)
    ax.axis('off')
plt.suptitle('CIFAR-10 sample (one per class) - labels shown only for the probe set')
plt.tight_layout(); plt.show()

In [ ]:
# === C08 . The single shared encoder (SmallCNN) ===
FEATURE_DIM = 128

class SmallCNN(nn.Module):
    '''Lightweight conv encoder: 32x32x3 -> feature vector of size feature_dim.
    Emits L2-UNnormalized features and has NO classification head.'''
    def __init__(self, feature_dim: int = FEATURE_DIM):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            block(3, 32),            # 32 -> 16
            block(32, 64),           # 16 -> 8
            block(64, feature_dim),  # 8 -> 4
        )
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 3:
            x = x.unsqueeze(0)
        assert x.dim() == 4 and x.shape[1] == 3, f'expected (N,3,H,W), got {tuple(x.shape)}'
        h = self.features(x)
        h = self.pool(h).flatten(1)
        return h

def make_encoder(seed: int = 0) -> SmallCNN:
    '''Build a fresh SmallCNN with reproducible init, placed on DEVICE.'''
    set_seed(seed)
    return SmallCNN(FEATURE_DIM).to(DEVICE)

# --- sanity checks ---
enc = make_encoder(0)
out = enc(torch.randn(2, 3, 32, 32).to(DEVICE))
assert out.shape == (2, FEATURE_DIM)
w0 = make_encoder(0).features[0][0].weight
w0b = make_encoder(0).features[0][0].weight
w1 = make_encoder(1).features[0][0].weight
assert torch.allclose(w0, w0b) and not torch.allclose(w0, w1)
n_params = sum(p.numel() for p in enc.parameters())
print(f'SmallCNN ready | FEATURE_DIM={FEATURE_DIM} | params={n_params:,} '
      f'(one architecture reused by random / rotation / SimCLR / collapse for a fair comparison)')

## 7 - The linear evaluation protocol: our single yardstick

How do we decide whether a pretext task actually learned anything useful? The standard SSL answer is the **linear probe**:

1. **Freeze** the pre-trained encoder (no gradients; eval mode, so BatchNorm uses running stats).
2. Extract features for a small labeled set.
3. Train **only a single linear layer** (logistic regression) on those frozen features.
4. Report the linear head's **test accuracy**.

```
[frozen encoder] --> features --> [trainable Linear(FEATURE_DIM -> 10)] --> class
     (no grad)                          (the only thing that learns)
```

Because the head is linear, its accuracy measures how **linearly separable** - i.e. how *immediately useful* - the representation is. This is the metric reported in the SimCLR / MoCo / BYOL papers, and the natural way to ask 'did this pretext task learn anything?'. We define it **now** (C10) because we reuse the exact same routine to score the random, rotation, and SimCLR encoders.

In [ ]:
# === C10 . The reusable linear-eval primitive + the random baseline ===
@torch.no_grad()
def _extract_features(encoder, images, batch_size=256):
    '''Frozen-encoder features for images ([N,3,32,32] in [0,1]) -> (N, FEATURE_DIM) on CPU.
    Applies Normalize(MEAN, STD) at the encoder input (the single documented point).'''
    encoder.eval()
    feats = []
    for i in range(0, images.shape[0], batch_size):
        batch = normalize(images[i:i + batch_size].to(DEVICE))
        feats.append(encoder(batch).cpu())
    return torch.cat(feats, 0)

def linear_probe(encoder, train_data=None, test_data=None, epochs=100, lr=1e-2, per_class=None):
    '''Freeze the encoder, extract features, train ONLY a Linear head, return top-1 test acc.'''
    if train_data is None:
        train_data = probe_train
    if test_data is None:
        test_data = probe_test
    encoder.requires_grad_(False)                 # never train the encoder

    train_imgs, train_lbls = train_data
    test_imgs, test_lbls = test_data

    # optional class-balanced subsample of the TRAIN set (used by C23)
    if per_class is not None:
        gp = torch.Generator().manual_seed(1000 + per_class)
        sel = []
        for c in range(10):
            idx = (train_lbls == c).nonzero(as_tuple=True)[0]
            take = min(per_class, idx.numel())
            if take < per_class:
                print('[warn] class ' + str(c) + ': only ' + str(take) + ' labeled; clamped.')
            sel.append(idx[torch.randperm(idx.numel(), generator=gp)[:take]])
        sel = torch.cat(sel)
        train_imgs, train_lbls = train_imgs[sel], train_lbls[sel]

    feat_train = _extract_features(encoder, train_imgs)
    feat_test = _extract_features(encoder, test_imgs)

    # standardize using TRAIN statistics (eps-guarded)
    mu = feat_train.mean(0, keepdim=True)
    sd = feat_train.std(0, keepdim=True) + 1e-6
    feat_train = ((feat_train - mu) / sd).to(DEVICE)
    feat_test = ((feat_test - mu) / sd).to(DEVICE)
    train_lbls, test_lbls = train_lbls.to(DEVICE), test_lbls.to(DEVICE)

    set_seed(0)                                   # reproducible head init + training
    head = nn.Linear(FEATURE_DIM, 10).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        F.cross_entropy(head(feat_train), train_lbls).backward()
        opt.step()

    with torch.no_grad():
        acc = (head(feat_test).argmax(1) == test_lbls).float().mean().item()
    return acc

encoder_random = make_encoder(seed=0)             # untrained: the no-pretraining floor
acc_random = linear_probe(encoder_random)
assert 0.0 <= acc_random <= 1.0
assert all(not p.requires_grad for p in encoder_random.parameters())   # encoder stays frozen
print(f'random-init linear-probe acc = {acc_random:.3f}')

## 3 - Pretext task #1: rotation prediction (RotNet)

The recipe (Gidaris et al., 2018): rotate each unlabeled image by **0, 90, 180, or 270 degrees** and train the encoder (plus a small 4-way head) to **predict which rotation was applied**. The label is generated for free - we *chose* the rotation.

**Why does predicting rotation teach anything?** To tell an upright dog from an upside-down one, the encoder must recognize the object's parts, their arrangement, and its canonical pose. Solving the silly-looking rotation task therefore forces genuinely useful structure into the features.

It is the canonical teaching example because it is **fully runnable**: a 4-way classification - no negatives, no pixel generation, no momentum encoders. **Prediction:** the rotation-pretrained encoder will linear-probe well *above* random init.

In [ ]:
# === C12 . The rotation pretext signal ===
def make_rotations(images):
    '''Given (B,3,32,32) return (4B,3,32,32) of the four 90-deg rotations and labels {0,1,2,3}.
    k counts 90-deg counter-clockwise turns: 0->0, 1->90, 2->180, 3->270 degrees.'''
    if images.dim() != 4:
        raise ValueError(f'expected 4D (B,3,H,W), got {tuple(images.shape)}')
    rotated, labels = [], []
    for k in range(4):
        rotated.append(torch.rot90(images, k, dims=(2, 3)))
        labels.append(torch.full((images.shape[0],), k, dtype=torch.long, device=images.device))
    return torch.cat(rotated, 0).contiguous(), torch.cat(labels, 0)

# --- sanity checks ---
_imgs, _labels = make_rotations(X_unlabeled[:8])
assert _imgs.shape == (32, 3, 32, 32) and _labels.shape == (32,)
assert set(_labels.tolist()) == {0, 1, 2, 3}
assert torch.allclose(torch.rot90(X_unlabeled[:1], 4, dims=(2, 3)), X_unlabeled[:1])  # 4 turns = identity

# --- visualize one image at all four rotations ---
sample = X_unlabeled[0]
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for k, ax in enumerate(axes):
    ax.imshow(_to_display(torch.rot90(sample, k, dims=(1, 2))))
    ax.set_title(f'{k * 90} deg  (label {k})')
    ax.axis('off')
plt.suptitle('Rotation pretext: the encoder must classify which rotation was applied')
plt.tight_layout(); plt.show()
rotation_demo_figure = fig

In [ ]:
# === C13 . Train the rotation-prediction model on the unlabeled pool ===
set_seed(1)
encoder_rot = make_encoder(seed=1)
rot_head = nn.Linear(FEATURE_DIM, 4).to(DEVICE)

ROT_EPOCHS = 8          # reduce on CPU if slow
ROT_BATCH = 128
opt = torch.optim.Adam(list(encoder_rot.parameters()) + list(rot_head.parameters()), lr=1e-3)

rotation_history = {'loss': [], 'acc': []}
encoder_rot.train()
N = X_unlabeled.shape[0]
for epoch in range(ROT_EPOCHS):
    order = torch.randperm(N)
    ep_loss, ep_correct, ep_count = 0.0, 0, 0
    for i in range(0, N, ROT_BATCH):
        idx = order[i:i + ROT_BATCH]
        if idx.numel() < 2:            # BatchNorm needs >= 2
            continue
        batch = X_unlabeled[idx].to(DEVICE)
        rot_imgs, rot_labels = make_rotations(batch)
        try:
            logits = rot_head(encoder_rot(normalize(rot_imgs)))
            loss = F.cross_entropy(logits, rot_labels)
            opt.zero_grad(); loss.backward(); opt.step()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                print('[warn] CUDA OOM - reduce ROT_BATCH or N_UNLABELED and re-run.')
            raise
        ep_loss += loss.item() * rot_imgs.shape[0]
        ep_correct += (logits.argmax(1) == rot_labels).sum().item()
        ep_count += rot_imgs.shape[0]
    l = ep_loss / ep_count
    a = ep_correct / ep_count
    rotation_history['loss'].append(l)
    rotation_history['acc'].append(a)
    print(f'epoch {epoch + 1:2d}/{ROT_EPOCHS} | loss {l:.3f} | rot-acc {a:.3f}')

# --- sanity checks ---
assert len(rotation_history['loss']) == ROT_EPOCHS
assert all(np.isfinite(v) for v in rotation_history['loss'])

# --- training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
xs = range(1, ROT_EPOCHS + 1)
ax1.plot(xs, rotation_history['loss'], marker='o'); ax1.set_title('Pretext loss'); ax1.set_xlabel('epoch')
ax2.plot(xs, rotation_history['acc'], marker='o', color='green')
ax2.axhline(0.25, ls='--', color='gray', label='chance (1/4)')
ax2.set_title('Rotation accuracy'); ax2.set_xlabel('epoch'); ax2.legend()
plt.tight_layout(); plt.show()

del rot_head            # keep only encoder_rot
a_final = rotation_history['acc'][-1]
print(f'final rotation-classification accuracy = {a_final:.3f} (chance = 0.25)')

In [ ]:
# === C14 . Measure the rotation representation ===
assert 'encoder_rot' in globals(), 'Run C13 first to create encoder_rot.'
acc_rotation = linear_probe(encoder_rot)
assert 0.0 <= acc_rotation <= 1.0

gain = acc_rotation - acc_random
print(f'random   linear-probe acc = {acc_random:.3f}')
print(f'rotation linear-probe acc = {acc_rotation:.3f}')
print(f'gain over baseline        = {gain:+.3f}  ({gain * 100:+.1f} pts)')
if acc_rotation < acc_random:
    print('[note] rotation < random this run - expected occasionally at this tiny scale '
          '(few epochs, small subset). Try re-running or increasing ROT_EPOCHS.')
else:
    print('First measurable SSL result: a label-free pretext task produced linearly-useful features.')

## 4 - Pretext task #2: contrastive learning (SimCLR)

**Core idea:** take an image, make **two random augmented views** of it - these form a **positive pair** we want *close* in embedding space. Every *other* image in the batch is a **negative** we want *far*. No labels needed; 'same image or not' is the free supervision.

**Augmentation defines the positive pair.** SimCLR's headline finding is that *strong* augmentation - random-resized crop, color distortion, Gaussian blur - is what makes the task hard enough to force good features. We make this tangible with sliders in C16.

**The loss (NT-Xent).** With L2-normalized projections and cosine similarity, for a batch of N images (-> 2N views) each view's positive is its partner; the loss is a softmax cross-entropy that pulls the positive together against all other views:

$$\mathcal{L}_{i,j} = -\log \frac{\exp(\mathrm{sim}(z_i, z_j)/\tau)}{\sum_{k \neq i} \exp(\mathrm{sim}(z_i, z_k)/\tau)}$$

where $\mathrm{sim}(u,v)$ is cosine similarity and $\tau$ is the temperature. A small MLP **projection head** is used only during pre-training and then discarded. **Prediction:** SimCLR probes at or above rotation.

In [ ]:
# === C16 . SimCLR augmentation pipeline + interactive explorer ===
def simclr_augment(images, crop_scale=0.5, jitter_strength=0.5, blur_sigma=1.0):
    '''Stochastic SimCLR view of [0,1] float images (B,3,32,32) -> [0,1] (B,3,32,32).
    Two independent calls give the two views. Normalize is applied later (encoder input).'''
    if images.dim() == 3:
        images = images.unsqueeze(0)
    crop_scale = float(min(max(crop_scale, 1e-2), 1.0))
    jitter_strength = float(max(jitter_strength, 0.0))
    blur_sigma = float(max(blur_sigma, 0.1))
    s = jitter_strength
    pipeline = transforms.Compose([
        transforms.RandomResizedCrop(32, scale=(crop_scale, 1.0), antialias=True),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.ColorJitter(0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s)], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, blur_sigma))], p=0.5),
    ])
    return torch.stack([pipeline(img) for img in images])

# --- sanity checks ---
_v = simclr_augment(X_unlabeled[:4])
assert _v.shape == (4, 3, 32, 32) and torch.isfinite(_v).all()
assert not torch.allclose(simclr_augment(X_unlabeled[:4]), simclr_augment(X_unlabeled[:4]))  # stochastic

SIMCLR_VIEW_SAMPLE = X_unlabeled[0]

def _show_views(crop_scale=0.5, jitter_strength=0.5, blur_sigma=1.0):
    v1 = simclr_augment(SIMCLR_VIEW_SAMPLE, crop_scale, jitter_strength, blur_sigma)[0]
    v2 = simclr_augment(SIMCLR_VIEW_SAMPLE, crop_scale, jitter_strength, blur_sigma)[0]
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    for ax, img, t in zip(axes, [SIMCLR_VIEW_SAMPLE, v1, v2], ['original', 'view 1', 'view 2']):
        ax.imshow(_to_display(img)); ax.set_title(t); ax.axis('off')
    plt.suptitle(f'crop_scale={crop_scale:.2f}  jitter={jitter_strength:.2f}  blur={blur_sigma:.2f}')
    plt.tight_layout(); plt.show()

augmentation_explorer = None
if WIDGETS_OK:
    try:
        augmentation_explorer = widgets.interact(
            _show_views,
            crop_scale=widgets.FloatSlider(value=0.5, min=0.2, max=1.0, step=0.05, description='crop'),
            jitter_strength=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='jitter'),
            blur_sigma=widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description='blur'),
        )
    except Exception as e:
        print(f'[warn] widget init failed ({e}); showing static default.')
        _show_views()
else:
    _show_views()

In [ ]:
# === C17 . Implement and run SimCLR (heaviest cell - GPU recommended) ===
def nt_xent_loss(z1, z2, temperature=0.5):
    '''Normalized temperature-scaled cross-entropy (NT-Xent) over 2N in-batch views.'''
    N = z1.shape[0]
    z = F.normalize(torch.cat([z1, z2], 0), dim=1)        # (2N, D)
    sim = (z @ z.t()) / temperature                       # (2N, 2N)
    sim.fill_diagonal_(-9e15)                             # mask self-similarity (not -inf)
    targets = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)]).to(z.device)
    return F.cross_entropy(sim, targets)

# quick loss sanity check
_z = torch.randn(4, 16)
assert torch.isfinite(nt_xent_loss(_z, _z + 0.01)).all()

set_seed(2)
encoder_simclr = make_encoder(seed=2)
proj = nn.Sequential(nn.Linear(FEATURE_DIM, 128), nn.ReLU(), nn.Linear(128, 64)).to(DEVICE)

SIMCLR_EPOCHS = 8           # fewer on CPU
SIMCLR_BATCH = 128          # negatives scale with batch size
SIMCLR_TEMPERATURE = 0.5
opt = torch.optim.Adam(list(encoder_simclr.parameters()) + list(proj.parameters()), lr=1e-3)

simclr_history = {'loss': []}
encoder_simclr.train()
N = X_unlabeled.shape[0]
for epoch in range(SIMCLR_EPOCHS):
    order = torch.randperm(N)
    ep_loss, ep_count = 0.0, 0
    for i in range(0, N, SIMCLR_BATCH):
        idx = order[i:i + SIMCLR_BATCH]
        if idx.numel() < 2:            # need >= 2 for the 2N view layout / BatchNorm
            continue
        batch = X_unlabeled[idx]
        v1 = normalize(simclr_augment(batch).to(DEVICE))
        v2 = normalize(simclr_augment(batch).to(DEVICE))
        try:
            z1, z2 = proj(encoder_simclr(v1)), proj(encoder_simclr(v2))
            loss = nt_xent_loss(z1, z2, SIMCLR_TEMPERATURE)
            opt.zero_grad(); loss.backward(); opt.step()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                print('[warn] CUDA OOM - reduce SIMCLR_BATCH or N_UNLABELED and re-run.')
            raise
        ep_loss += loss.item() * idx.numel(); ep_count += idx.numel()
    ep_mean = ep_loss / ep_count
    simclr_history['loss'].append(ep_mean)
    print(f'epoch {epoch + 1:2d}/{SIMCLR_EPOCHS} | NT-Xent loss {ep_mean:.3f}')

assert all(np.isfinite(v) for v in simclr_history['loss'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, SIMCLR_EPOCHS + 1), simclr_history['loss'], marker='o')
ax.set_title('SimCLR NT-Xent loss'); ax.set_xlabel('epoch'); ax.set_ylabel('loss')
plt.tight_layout(); plt.show()

del proj                    # keep only encoder_simclr
first, final = simclr_history['loss'][0], simclr_history['loss'][-1]
trend = 'decreased' if final < first else 'did not decrease'
print(f'final NT-Xent loss = {final:.3f} ({trend} over training)')
if DEVICE.type == 'cpu':
    print('[note] On CPU this is the slowest cell; reduce SIMCLR_EPOCHS / SIMCLR_BATCH if needed.')

In [ ]:
# === C18 . Measure the SimCLR representation ===
assert 'encoder_simclr' in globals(), 'Run C17 first to create encoder_simclr.'
acc_simclr = linear_probe(encoder_simclr)
assert 0.0 <= acc_simclr <= 1.0

print(f'random   linear-probe acc = {acc_random:.3f}')
print(f'rotation linear-probe acc = {acc_rotation:.3f}  ({acc_rotation - acc_random:+.3f})')
print(f'simclr   linear-probe acc = {acc_simclr:.3f}  ({acc_simclr - acc_random:+.3f})')
if acc_simclr < acc_random:
    print('[note] simclr < random this run - small-scale training variance '
          '(few epochs, small batch -> few negatives). Increase SIMCLR_EPOCHS / SIMCLR_BATCH.')

## 5 and 6 - Bootstrapping (BYOL / SimSiam) and the collapse problem

Contrastive learning needs **negatives**. Bootstrapping methods - **BYOL**, **SimSiam** - drop them entirely and learn from **positive pairs only**: just pull the two views of an image together. But that invites a catastrophe.

**Feature collapse.** If the only objective is 'make the two views' embeddings match', the encoder can cheat by mapping *every* input to the **same constant vector**. Then the loss is a perfect zero and the representation is **useless** - every image looks identical.

**The asymmetric fix.** SimSiam prevents collapse with two architectural tricks on otherwise negative-free training:

- a **predictor head** on one branch (online), and
- a **stop-gradient** on the other branch (target): the target is treated as a fixed label, so no gradient flows back through it.

BYOL adds an **EMA teacher** whose weights are a moving average of the student - reframing SSL as **self-distillation**. The stop-gradient breaks the symmetry that makes the constant solution an attractor.

**The experiment (C20):** train a *naive symmetric* Siamese net (no stop-grad) and a *SimSiam-style* net (predictor + stop-grad) on the same positive pairs, and **watch the embedding variance** - the naive one collapses toward zero variance while SimSiam stays healthy.

In [ ]:
# === C20 . Demonstrate collapse and its prevention ===
COLLAPSE_STEPS = 150
COLLAPSE_BATCH = 128

def _train_siamese(mode, steps, batch_size, seed=3):
    '''Train a negative-free Siamese net; return per-step embedding-std (the collapse metric).'''
    if mode not in {'naive', 'simsiam'}:
        raise ValueError(f'mode must be naive or simsiam, got {mode}')
    set_seed(seed)
    enc = make_encoder(seed)
    proj = nn.Sequential(nn.Linear(FEATURE_DIM, 128), nn.ReLU(), nn.Linear(128, 64)).to(DEVICE)
    params = list(enc.parameters()) + list(proj.parameters())
    pred = None
    if mode == 'simsiam':
        pred = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 64)).to(DEVICE)
        params += list(pred.parameters())
    opt = torch.optim.Adam(params, lr=1e-3)

    enc.train()
    N = X_unlabeled.shape[0]
    std_history = []
    for step in range(steps):
        idx = torch.randperm(N)[:batch_size]
        if idx.numel() < 2:
            continue
        batch = X_unlabeled[idx]
        v1 = normalize(simclr_augment(batch).to(DEVICE))
        v2 = normalize(simclr_augment(batch).to(DEVICE))
        z1, z2 = proj(enc(v1)), proj(enc(v2))
        if mode == 'naive':
            # symmetric, NO stop-gradient: the collapse-prone objective
            loss = -F.cosine_similarity(z1, z2, dim=1).mean()
        else:
            # SimSiam: predictor on one branch + stop-gradient (.detach) on the target
            p1, p2 = pred(z1), pred(z2)
            loss = -(F.cosine_similarity(p1, z2.detach(), dim=1).mean()
                     + F.cosine_similarity(p2, z1.detach(), dim=1).mean()) / 2
        opt.zero_grad(); loss.backward(); opt.step()
        # collapse metric: std of L2-normalized embeddings across the batch (-> 0 means collapse)
        std_history.append(F.normalize(z1, dim=1).std(dim=0).mean().item())
    return std_history

collapse_history = {
    'naive': _train_siamese('naive', COLLAPSE_STEPS, COLLAPSE_BATCH),
    'simsiam': _train_siamese('simsiam', COLLAPSE_STEPS, COLLAPSE_BATCH),
}

assert len(collapse_history['naive']) == COLLAPSE_STEPS
assert len(collapse_history['simsiam']) == COLLAPSE_STEPS
assert all(np.isfinite(v) and v >= 0 for v in collapse_history['naive'] + collapse_history['simsiam'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(collapse_history['naive'], label='naive symmetric (collapses)', color='crimson')
ax.plot(collapse_history['simsiam'], label='SimSiam: predictor + stop-grad (stable)', color='seagreen')
ax.set_xlabel('training step'); ax.set_ylabel('embedding std (L2-normalized)')
ax.set_title('Feature collapse and its prevention'); ax.legend()
plt.tight_layout(); plt.show()
collapse_figure = fig

n_final, s_final = collapse_history['naive'][-1], collapse_history['simsiam'][-1]
print(f'final embedding std - naive: {n_final:.4f} | simsiam: {s_final:.4f}')
if n_final < s_final:
    print('As predicted: the naive net embeddings collapsed toward a constant; SimSiam stayed healthy.')
else:
    print('[note] ordering not as expected this run - run-to-run variance on tiny data; try re-running.')

## Interpreting collapse: every family avoids it differently

You just *watched* the central difficulty of negative-free SSL: without a force keeping embeddings spread out, the encoder collapses to a constant. Map this back to the C04 taxonomy - **collapse-avoidance is what distinguishes the families**:

| Family | Collapse-avoidance mechanism |
|---|---|
| **Contrastive** (SimCLR, MoCo) | **Negatives** - NT-Xent explicitly pushes other images apart, so a constant embedding is heavily penalized. |
| **Bootstrapping** (BYOL, SimSiam) | **Architectural asymmetry** - predictor + stop-gradient (+ EMA target) breaks the symmetry that makes the constant solution an attractor (the curve you just saw). |
| **Regularization** (Barlow Twins, VICReg) | **Explicit variance / covariance terms** - directly require each embedding dimension to stay varied and decorrelated. |
| **Predictive** (rotation) and **Generative** (MAE) | **A non-trivial target** - a constant output simply cannot predict the rotation / reconstruct the pixels, so collapse is not a solution in the first place. |

The whole lecture's spine - five families, one map - is now grounded in something you ran: each family is, at heart, a different answer to 'how do we stop the encoder from cheating?'.

In [ ]:
# === C22 . Head-to-head synthesis under the linear-eval protocol ===
for _n in ['acc_random', 'acc_rotation', 'acc_simclr']:
    assert _n in globals(), _n + ' missing - run C10 / C14 / C18 first.'

def _safe(x):
    if not np.isfinite(x):
        print('[warn] non-finite accuracy substituted with 0.0')
        return 0.0
    return float(x)

results_table = {'random': _safe(acc_random), 'rotation': _safe(acc_rotation), 'simclr': _safe(acc_simclr)}

print('{:<10}{:>10}{:>16}'.format('encoder', 'accuracy', 'gain vs random'))
print('-' * 36)
for name, acc in results_table.items():
    print('{:<10}{:>10.3f}{:>+16.3f}'.format(name, acc, acc - results_table['random']))

fig, ax = plt.subplots(figsize=(7, 4.5))
names = list(results_table.keys())
vals = [results_table[n] for n in names]
bars = ax.bar(names, vals, color=['#999999', '#4c72b0', '#dd8452'])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
ax.axhline(results_table['random'], ls='--', color='gray', label='random baseline')
ax.set_ylim(0, 1); ax.set_ylabel('linear-probe test accuracy')
ax.set_title('Representation quality by pretext task'); ax.legend()
plt.tight_layout(); plt.show()
comparison_figure = fig

print()
print('Takeaway: both label-free pretext tasks beat random init - the encoders learned '
      'transferable structure from unlabeled images alone.')

In [ ]:
# === C23 . Interactive label-scarcity experiment (the centerpiece) ===
PROBE_LABEL_BUDGET_DEFAULT = 10

# Cache frozen features ONCE per encoder so slider moves never re-run the encoders.
_encs = {'random': encoder_random, 'rotation': encoder_rot, 'simclr': encoder_simclr}
cached_features = {
    name: (_extract_features(e, probe_train[0]), probe_train[1],
           _extract_features(e, probe_test[0]), probe_test[1])
    for name, e in _encs.items()
}
assert set(cached_features) == {'random', 'rotation', 'simclr'}
assert all(cached_features[n][0].shape[1] == FEATURE_DIM for n in cached_features)

def _probe_from_cached(name, per_class, epochs=100, lr=1e-2):
    ftr, ytr, fte, yte = cached_features[name]
    gp = torch.Generator().manual_seed(1000 + per_class)
    sel = []
    for c in range(10):
        idx = (ytr == c).nonzero(as_tuple=True)[0]
        take = min(per_class, idx.numel())
        sel.append(idx[torch.randperm(idx.numel(), generator=gp)[:take]])
    sel = torch.cat(sel)
    ftr_s, ytr_s = ftr[sel], ytr[sel]
    mu = ftr_s.mean(0, keepdim=True); sd = ftr_s.std(0, keepdim=True) + 1e-6
    ftr_s = ((ftr_s - mu) / sd).to(DEVICE)
    fte_s = ((fte - mu) / sd).to(DEVICE)
    ytr_d, yte_d = ytr_s.to(DEVICE), yte.to(DEVICE)
    set_seed(0)
    head = nn.Linear(FEATURE_DIM, 10).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad(); F.cross_entropy(head(ftr_s), ytr_d).backward(); opt.step()
    with torch.no_grad():
        return (head(fte_s).argmax(1) == yte_d).float().mean().item()

_max_per_class = min((cached_features['random'][1] == c).sum().item() for c in range(10))

def redraw(n_per_class=PROBE_LABEL_BUDGET_DEFAULT):
    n_per_class = int(min(n_per_class, _max_per_class))
    names = ['random', 'rotation', 'simclr']
    accs = [_probe_from_cached(n, n_per_class) for n in names]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    bars = ax.bar(names, accs, color=['#999999', '#4c72b0', '#dd8452'])
    for b, v in zip(bars, accs):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
    ax.set_ylim(0, 1); ax.set_ylabel('linear-probe test accuracy')
    ax.set_title(f'Linear probe with {n_per_class} labeled example(s) / class')
    plt.tight_layout(); plt.show()
    if accs[1] >= accs[0] or accs[2] >= accs[0]:
        print(f'@ {n_per_class}/class: SSL >= random '
              f'(rotation {accs[1] - accs[0]:+.3f}, simclr {accs[2] - accs[0]:+.3f})')
    else:
        print(f'[note] @ {n_per_class}/class SSL < random this run (small-scale variance).')

interactive_probe_comparison = None
if WIDGETS_OK:
    try:
        interactive_probe_comparison = widgets.interact(
            redraw,
            n_per_class=widgets.IntSlider(value=PROBE_LABEL_BUDGET_DEFAULT, min=1,
                                          max=min(50, _max_per_class), step=1, description='labels/class'),
        )
    except Exception as e:
        print(f'[warn] widget init failed ({e}); showing static default.')
        redraw(PROBE_LABEL_BUDGET_DEFAULT)
else:
    redraw(PROBE_LABEL_BUDGET_DEFAULT)

## Synthesis and outlook

**The storyline, per learning objective:**

- **LO1** - SSL is *pretext-then-transfer*; we placed five families on one map (C03-C04).
- **LO2** - the **linear-eval protocol** gave us one fair yardstick, anchored by a random-init baseline (C09-C10).
- **LO3 / LO4** - two runnable pretext tasks, **rotation** (C11-C14) and **SimCLR** (C15-C18), both beat random init.
- **LO5** - we *watched* **feature collapse** and saw predictor + stop-gradient prevent it (C19-C21).
- **LO6** - the head-to-head (C22) and the **label-scarcity widget** (C23) showed SSL's advantage widening as labels get scarce - the practical payoff.

**Deliberately deferred (try these as extensions):**

- **Generative** - image-GPT, masked autoencoders (MAE): reconstruct pixels / patches.
- **Contrastive at scale** - MoCo's momentum encoder + negative queue (decouples the number of negatives from batch size).
- **Predictive variants** - relative-patch-position / jigsaw context tasks, DeepCluster clustering targets.
- **Regularization** - Barlow Twins, VICReg: explicit variance / covariance objectives.

**References (source slides + original papers):** Rotation / RotNet (Gidaris et al., 2018); SimCLR (Chen et al., 2020); MoCo (He et al., 2020); BYOL (Grill et al., 2020); SimSiam (Chen and He, 2021); Barlow Twins (Zbontar et al., 2021); VICReg (Bardes et al., 2022).

> Same recipe, one encoder, one yardstick: change only *how the encoder learns from unlabeled data*, and measure what it is worth. That is the heart of self-supervised representation learning.